__Exclude exposures affected by the "fiducials on" issue__

In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [3]:
cat = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/sv1-exposures_20210224.fits')
print(len(cat))

tileid_list = [80605, 80606, 80607, 80608, 80609, 80610, 80613]

mask = np.in1d(cat['TILEID'], tileid_list)
cat = cat[mask]
print(len(cat), len(np.unique(cat['EXPID'])))
cat.sort('EXPID')

cat[:1]

1243
161 161


NIGHT,EXPID,FIELD,TARGETS,OBSCONDITIONS,ARIZONA_TIMEOBS,EBV,SPECMODEL_SKY_GMAG_AB,SPECMODEL_SKY_RMAG_AB,SPECMODEL_SKY_ZMAG_AB,GFA_ORIGIN,B_DEPTH,R_DEPTH,Z_DEPTH,B_DEPTH_EBVAIR,R_DEPTH_EBVAIR,Z_DEPTH_EBVAIR,DAILY_BITPSFFN,DAILY_BITFRAMEFN,DAILY_BITSKYFN,DAILY_BITSFRAMEFN,DAILY_BITFLUXCALIBFN,DAILY_BITCFRAMEFN,TGT,SKY,STD,WD,LRG,ELG,QSO,BGS,MWS,TILEID,TILERA,TILEDEC,EXPTIME,MJDOBS,SKYMON_NEXP,SKYMON_SKYCAM0_MEAN,SKYMON_SKYCAM0_MEAN_ERR,SKYMON_SKYCAM1_MEAN,SKYMON_SKYCAM1_MEAN_ERR,SKYMON_AVERAGE_MEAN,SKYMON_AVERAGE_MEAN_ERR,GFA_AIRMASS,GFA_MOON_ILLUMINATION,GFA_MOON_ZD_DEG,GFA_MOON_SEP_DEG,GFA_TRANSPARENCY,GFA_FWHM_ASEC,GFA_SKY_MAG_AB,GFA_FIBER_FRACFLUX,GFA_FIBER_FRACFLUX_ELG,GFA_TRANSPFRAC,GFA_MAXCONTRAST,GFA_MINCONTRAST,GFA_KTERM,GFA_RADPROF_FWHM_ASEC,GFA_FIBERFAC,GFA_FIBERFAC_ELG,EPHEM_NOON,EPHEM_DUSK,EPHEM_DAWN,EPHEM_BRIGHTDUSK,EPHEM_BRIGHTDAWN,EPHEM_BRIGHTDUSK_LST,EPHEM_BRIGHTDAWN_LST,EPHEM_MOONRISE,EPHEM_MOONSET,EPHEM_MOON_ILLUM_FRAC,EPHEM_NEAREST_FULL_MOON
int64,int64,bytes30,bytes16,int16,bytes19,float32,float32,float32,float32,bytes13,float32,float32,float32,float32,float32,float32,int64,int64,int64,int64,int64,int64,int16,int16,int16,int16,int16,int16,int16,int16,int16,int64,float32,float32,float32,float32,int16,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
20201214,67710,XMM-LSS,QSO+LRG,1,2020-12-14T21:12:40,0.026,22.46653,21.36987,19.686352,matched_coadd,19.1,17.0,18.6,14.928702,14.590197,17.078018,1073741823,1073741823,1073741823,1073741823,0,0,4200,800,136,13,2117,1341,1428,464,191,80605,36.448,-4.601,900.0,59198.176,0,-99.0,-99.0,-99.0,-99.0,-99.0,-99.0,1.2455698,0.0036080289,133.7576,119.15718,0.8552955,3.5648944,20.274342,0.09248331,0.08644419,0.07910056,4.109576,3.6593802,0.114,3.7471519,0.1273114,0.1655635,59197.791666666664,59198.07704947222,59198.53638188851,59198.05633958238,59198.557098368896,-7.152731532294979,173.61402820025236,59197.59470830986,59198.02745986081,0.005432771706774475,14.852906347514363


In [4]:
# David Schlegel's list of exposures with speed == r_depth / exptime > 0.3
explist_dark = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/explist-deep-dark.txt', format='ascii')
explist_bright = Table.read('/global/cfs/cdirs/desi/users/rongpu/spectro/sv1/explist-deep-bright.txt', format='ascii')

explist = vstack([explist_dark, explist_bright], join_type='exact')
print(len(explist))

explist.sort('EXPID')

explist[:1]

96


TILEID,NIGHT,EXPID
int64,int64,int64
80607,20201214,67744


In [5]:
speed = cat['R_DEPTH'] / cat['EXPTIME']
mask = speed > 0.3
print(np.sum(mask))

np.all(explist['EXPID']==cat['EXPID'][mask])

cat = cat[mask]
print(len(cat))

96
96


In [6]:
# List of exposures actually included in cascades
explist = Table.read('/global/cfs/cdirs/desi/spectro/redux/cascades/run/scripts/tiles/explist-all-sv1.txt', format='ascii')

mask = np.in1d(cat['EXPID'], explist['EXPID'])
print(np.sum(mask))

print('Exposures excluded:')
cat[~mask][['NIGHT', 'EXPID', 'FIELD', 'TILEID', 'TARGETS']]

93
Exposures excluded:


NIGHT,EXPID,FIELD,TILEID,TARGETS
int64,int64,bytes30,int64,bytes16
20201222,69438,Lynx,80608,ELG
20201222,69440,Lynx,80607,QSO+LRG
20201222,69441,Lynx,80607,QSO+LRG


In [7]:
cat = cat[mask]
print(len(cat))

93


In [8]:
# Only rerun the affected exposures
mask = np.in1d(cat['TILEID'], [80607, 80608])
cat = cat[mask]
print(len(cat))

34


In [9]:
for tileid in np.unique(cat['TILEID']):
    mask = cat['TILEID']==tileid
    print(tileid, np.sum(cat['R_DEPTH_EBVAIR'][mask]), np.sum(mask), cat['TARGETS'][mask][0], cat['FIELD'][mask][0])

80607 9416.092 15 QSO+LRG Lynx
80608 13686.26 19 ELG Lynx


In [10]:
for tileid in np.unique(cat['TILEID']):
    mask = cat['TILEID']==tileid
    print(tileid, np.sum(cat['R_DEPTH_EBVAIR'][mask]), np.sum(cat['R_DEPTH_EBVAIR'][mask])/4000.)

80607 9416.092 2.35402294921875
80608 13686.26 3.42156494140625


In [11]:
print('TILEID n_exp tot_depth n_sub_min n_sub_max  n_sub')
for tileid in np.unique(cat['TILEID']):
    if tileid==80613:
        nom_depth = 600.
    else:
        nom_depth = 4000.
    mask = cat['TILEID']==tileid
    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    n_sub_min = tot_depth/nom_depth
    if n_sub_min < 2:
        margin = 0.15
    else:
        margin = 0.1
    n_sub_max = tot_depth/(nom_depth*(1-margin))
    n_sub = int(np.floor(n_sub_max))
    print(tileid, '{:3} {:9.2f} {:6.2f} {:6.2f} {:3}'.format(np.sum(mask), tot_depth, n_sub_min, n_sub_max, n_sub))

TILEID n_exp tot_depth n_sub_min n_sub_max  n_sub
80607  15   9416.09   2.35   2.62   2
80608  19  13686.26   3.42   3.80   3


----

In [17]:
np.random.seed(125)

subsamp_dict = {}

# tileid = 80607
for tileid in np.unique(cat['TILEID']):
    
    print()
    
    if tileid==80613:
        nom_depth = 600.
    else:
        nom_depth = 4000.
    mask = cat['TILEID']==tileid

    n_exp = np.sum(mask)

    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    n_sub_min = tot_depth/nom_depth
    if n_sub_min < 2:
        margin = 0.15
    else:
        margin = 0.1
    n_sub_max = tot_depth/(nom_depth*(1-margin))
    n_sub = int(np.floor(n_sub_max))
    print(tileid, '{:3} {:9.2f} {:6.2f} {:6.2f} {:3}'.format(np.sum(mask), tot_depth, n_sub_min, n_sub_max, n_sub))

    mask = cat['TILEID']==tileid
    expid_list = np.array(cat['EXPID'][mask])
    subsets = []

    total_iteration = 0
    while True:
        total_iteration += 1
        success = False
        iteration = 0
        while success==False:
            iteration += 1
            subset = []
            subset_depth = []
            while True:
                mask = ~np.in1d(expid_list, subset)
                expid = np.random.choice(expid_list[mask])
                subset.append(expid)
                subset_depth.append(cat['R_DEPTH_EBVAIR'][cat['EXPID']==expid][0])

                if total_iteration<200:
                    margin = 0.05
                elif ((total_iteration>200) & (total_iteration<400)):
                    margin = 0.1
                elif (total_iteration>400):
                    margin = 0.15
                if (np.sum(subset_depth)>(1-margin)*nom_depth) and (np.sum(subset_depth)<(1+margin)*nom_depth):
    #             if (np.sum(subset_depth)>0.8*nom_depth) and (np.sum(subset_depth)<1.2*nom_depth):
                    success = True
                    # print(np.sum(subset_depth))
                    break
                if len(subset)==len(expid_list):
                    break
            if iteration>(np.minimum(100, np.math.factorial(n_exp))):
                # reset
                mask = cat['TILEID']==tileid
                expid_list = np.array(cat['EXPID'][mask])
                subsets = []
                break
        if success:
            subsets.append(subset)
            mask = ~np.in1d(expid_list, np.concatenate(subsets))
            expid_list = expid_list[mask]
        if len(subsets)==n_sub:
            break
    
    print('tile', tileid, subsets)
    subsamp_dict[str(tileid)] = subsets
    for subset in subsets:
        mask = np.in1d(cat['EXPID'], subset)
        print(np.sum(mask), np.sum(cat['R_DEPTH_EBVAIR'][mask]))


80607  15   9416.09   2.35   2.62   2
tile 80607 [[67767, 68028, 68847, 68664, 68845, 68846], [67765, 68662, 68663, 68027, 67766, 68666]]
6 4054.5356
6 3969.8909

80608  19  13686.26   3.42   3.80   3
tile 80608 [[68842, 69252, 68023, 68024, 68661], [69253, 68491, 67769, 69249, 67770, 68328, 67771], [68660, 68327, 68317, 68025, 68841]]
5 3884.9497
7 4011.8179
5 4095.446


In [19]:
subsamp_dict

{'80607': [[67767, 68028, 68847, 68664, 68845, 68846],
  [67765, 68662, 68663, 68027, 67766, 68666]],
 '80608': [[68842, 69252, 68023, 68024, 68661],
  [69253, 68491, 67769, 69249, 67770, 68328, 67771],
  [68660, 68327, 68317, 68025, 68841]]}

In [20]:
# Print summary
for tileid_str in subsamp_dict.keys():
    tileid = int(tileid_str)
    mask = cat['TILEID']==tileid
    n_exp = np.sum(mask)
    tot_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    print('Tile {}:'.format(tileid))
    print('all:       n_exp={:2}  depth={:5.0f}s'.format(n_exp, tot_depth))
    subsets = subsamp_dict[tileid_str]
    for index, subset in enumerate(subsets):
        mask = np.in1d(cat['EXPID'], subset)
        subset_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
        print('subset {}:  n_exp={:2}  depth={:5.0f}s'.format(index+1, len(subset), subset_depth))
    mask = (cat['TILEID']==tileid) & (~np.in1d(cat['EXPID'], np.concatenate(subsets)))
    unused_depth = np.sum(cat['R_DEPTH_EBVAIR'][mask])
    print('unused:    n_exp={:2}  depth={:5.0f}s'.format(np.sum(mask), unused_depth))
    print()

Tile 80607:
all:       n_exp=15  depth= 9416s
subset 1:  n_exp= 6  depth= 4055s
subset 2:  n_exp= 6  depth= 3970s
unused:    n_exp= 3  depth= 1392s

Tile 80608:
all:       n_exp=19  depth=13686s
subset 1:  n_exp= 5  depth= 3885s
subset 2:  n_exp= 7  depth= 4012s
subset 3:  n_exp= 5  depth= 4095s
unused:    n_exp= 2  depth= 1694s

